In [1]:
import scipy as sp
import numpy as np
import copy
import numpy.random as rnd

The gradient (and loss) as a function :

In [27]:
def GradERM(X, y, w, v, LambdaRegularization):
    zw = np.matmul(X, w)
    zv = np.matmul(X, v)
    GradwTotal = 2*np.matmul(np.transpose(X), (zw - y)) + LambdaRegularization*w
    GradvTotal = LambdaRegularization*v
    return(np.stack((GradwTotal, GradvTotal), axis = 1))

def LossERM(X, y, w, v, LambdaRegularization):
    zw = np.matmul(X, w)
    zv = np.matmul(X, v)
    return( np.sum(np.power((y - zw),2), 0)/X.shape[0] + LambdaRegularization*( np.dot(w,w) + np.dot(v,v) )/2 )

The GD step

In [3]:
def GDStepERM(X, y, wv, LambdaRegularization, LearningRate):
    wv -= LearningRate*GradERM(X, y, wv[:,0], wv[:,1], LambdaRegularization)
    return(LossERM(X, y, wv[:,0], wv[:,1], LambdaRegularization))

Full GD function

In [ ]:
def GDERM(X, y, wv, LambdaRegularization = 1, LearningRate = 0.02, MaxIter = 1e4, EpsConvergence = 1e-6, Verbose = True, VerboseRate = 100):
    Conv = 1
    NIter = 0
    Losses = [LossERM(X, y, wv[:,0], wv[:,1], LambdaRegularization)]
    print("Iteration %s" % NIter)
    print("Current loss %s" % Losses[NIter])
    while((NIter < MaxIter) and (Conv > EpsConvergence)):
        Losses.append(GDStepERM(X, y, wv, LambdaRegularization, LearningRate))
        NIter += 1
        Conv = np.abs(Losses[NIter] - Losses[NIter-1])/np.abs(Losses[NIter])
        if(Verbose and NIter%VerboseRate == 0):
            print("Iteration %s" % NIter)
            print("Current loss %s" % Losses[NIter])
            print("Current convergence criterion %s" % Conv)
    print("Iteration %s" % NIter)
    print("Current loss %s" % Losses[NIter])
    print("Current convergence criterion %s" % Conv)
    return(np.array(Losses))

Main variables 

In [48]:
d = 400
M = 400
LearningRate = 0.002
LambdaRegularization = 1

In [49]:
X = rnd.normal(0, 1/np.sqrt(d), size = (M, d))
wvTrue = rnd.normal(0, 1, size = (d, 2))
wvLearned = rnd.normal(0, 1, size = (d, 2))
yTrue = np.matmul(X, wvTrue[:,0])
alphas = M/d
GDERM(X, yTrue, wvLearned, LearningRate=LearningRate, LambdaRegularization=LambdaRegularization, MaxIter = 100000, VerboseRate = 5000, EpsConvergence=1e-10)
cosw = (np.dot(wvLearned[:,0], wvTrue[:,0]))/d
cosv = (np.dot(wvLearned[:,1], wvTrue[:,1]))/d
normw = np.dot(wvLearned[:,0], wvLearned[:,0])/d
normv = np.dot(wvLearned[:,1], wvLearned[:,1])/d
normw0 = np.dot(wvTrue[:,0], wvTrue[:,0])/d
normv0 = np.dot(wvTrue[:,1], wvTrue[:,1])/d

Iteration 0
Current loss 365.3094471181332
Iteration 3720
Current loss 61.81180255098964
Current convergence criterion 9.713356679886094e-11


In [50]:
print(cosw)
print(cosv)
print(normw)
print(normv)
print(normw0)
print(normv0)

0.4772583301073328
9.951533483277745e-06
0.30863717780353217
3.0971296373612085e-07
1.0167360158543177
0.9331895078816258


In [51]:
XTest = rnd.normal(0, 1/np.sqrt(d), size = (M, d))
yTest = np.matmul(XTest, wvTrue[:,0])
yTrial = np.matmul(XTest, wvLearned[:,0])
np.mean(np.square(yTrial-yTest))

0.33367572352878966